# Wideband polarimetry


In [ ]:
#Reset the session for each pulsar

%reset -f

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.gridspec as gridspec


#PULSEPORTRAITURE
import pplib_pol as ppl
import ppspline_pol as pps  

#PSRCHIVE
import psrchive as pr


plt.rcParams.update({'font.size': 14})
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "Times New Roman"

## Set up your working directories and files 

In [ ]:
psr = 'J0125-2327'  

datadir   = f'data/'
resultdir = f'results/{psr}'     

os.system(f'mkdir -p {resultdir}')

## Run the analysis

### Load the polarimetric portraits

In [ ]:
portrait      = f'{datadir}/{psr}.port1dm.pazi8'

In [ ]:
## Load polarimetric data portrait
dp = pps.DataPortrait(portrait, state='Stokes')      #state is either 'Intensity' or 'Stokes'

In [ ]:
## Calculate basic polarimetry (L, fractions, PAs)
## Note: on/off pulse mask which is calculated in polarimetry fails for profiles wrapped at the edges, 
## use phase_rot to avoid wrapping and help finding correct onpulse mask. Rotation is used only for finding the mask, not applied to data. 

phase_rot = 0.2

dp.polarimetry(phase_rot=phase_rot,quiet=False)

#### Show portraits

In [ ]:
dp.show_data_portrait(state='Stokes')   # state is either 'Intensity', 'Stokes' or 'Stokes_L' (Stokes_L requires dp.polarimetry(), Stokes doesn't)

In [ ]:
dp.show_data_portrait(state='Stokes_L') # state is either 'Intensity', 'Stokes' or 'Stokes_L' (Stokes_L requires dp.polarimetry(), Stokes doesn't)

In [ ]:
#Save portrait plots
dp.show_data_portrait(state='Stokes', savefig=True, outname=f'{psr}_IQUV', resultdir=f'{resultdir}') 
dp.show_data_portrait(state='Stokes_L', savefig=True, outname=f'{psr}_ILV', resultdir=f'{resultdir}') 

In [ ]:
# dp.polarimetry() is required; defualt aspect=[nrow=8,ncol=1,nbin=dp.nbin] and figsize=(6,15) works best for 8 channels, change at will
# use phase_rot to rotate the profiles in the plot (data arrays are not rotated)
# access data arrays for I, L and V with dp.portL (np.shape(nsub=1,npol,nchan,nbin))

dp.show_pol_profiles(phase_rot=phase_rot) 

#### Check onpulse mask

In [ ]:
# dp.polarimetry() is required
# select mtype='average' for frequency-average profile and its mask or mtype='chan' to see on pulse bins per channel
# if mtype='chan': default plot aspect=[nrow=8,ncol=1] and figsize=(6,15) works best for 8 channels, change at will
# access onpulse masks with dp.onpulse_mask (average profile) and dp.onpulse_mask_f (per channel)

dp.show_onpulse_mask(mtype='chan') 

#### Get pulse widths

In [ ]:
#Pulse widths at each frequency as a sum of on-pulse bins
pulse_width_bin = dp.onpulse_mask_f.sum(axis=1) #in bins
pulse_width_rot = pulse_width_bin * 360/dp.nbin #in deg

### Flux

#### Phase-averaged spectrum from the portrait

In [ ]:
dp.fit_flux_profile(state='Stokes')

#### Phase-resolved spectral index from the portrait

In [ ]:
#Access phase-resolve spectral indices as data arrays:
#dp.alphas_ph, dp.alphas_ph_err

dp.phase_resolved_alpha(plot=True, err_th=0.5, phase_rot=phase_rot)


### Polarisation fractions

#### Plot phase-averaged fractions

In [ ]:
dp.show_fractions(snr_th=1, savefig=False, resultdir='.')

#### Waterfall plots (phase- and frequency-resolved fractions)

In [ ]:
dp.show_waterfall_plot(phase_rot=phase_rot, rangeL=[0,1], rangeV=[-0.3,0.3])

### Final plot (PAs, average profiles, phase-resolved spectral indices)

In [ ]:
profI = dp.Lflux_prof[0,0]
profL = dp.Lflux_prof[0,1]
profV = dp.Lflux_prof[0,2]

alphas, alpha_errs = dp.alphas_ph, dp.alphas_ph_err

phase = np.linspace(0, 1, dp.nbin, endpoint=False)

# Extract channel × bin arrays consistently
portL = dp.portLx[0,0]               # (nchan, nbin)
noise = dp.Lnoise_stdsxs[0,0]        # (nchan, nbin)
freqs = dp.freqsxs[0]                # (nchan,)

# ===========================
# Create figure + subplots
# ===========================
fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1, figsize=(6, 10),
    sharex=True, gridspec_kw={"hspace": 0.0})

# ======================================
# 1) PA plot (with masks)
# ======================================
for i, freq in enumerate(freqs):

    Lmask = portL[i] > 10 * noise[i]
    sigmask = dp.PAx_ph_sigma[i] < 0.5
    m = Lmask & sigmask

    # Convert to degrees
    PA_deg = np.degrees(dp.PAx_ph[i][m])
    PA_err_deg = np.degrees(dp.PAx_ph_sigma[i][m])

    # Shifted phase
    phase_shifted = (phase[m] - phase_rot) % 1.0

    ax1.errorbar(
        phase_shifted, PA_deg, yerr=PA_err_deg,
        fmt='o', ms=3,
        label=f"{freq:.1f} MHz"
    )

ax1.set_ylabel("PA [deg]")
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=8)

# ======================================
# 2) Pulse profiles I, L, V
# ======================================
totI_rot = ppl.rotate_profile(profI, phase=phase_rot)
totL_rot = ppl.rotate_profile(profL, phase=phase_rot)
totV_rot = ppl.rotate_profile(profV, phase=phase_rot)

ax2.plot(phase, totI_rot, lw=1, color='black', label='I')
ax2.plot(phase, totL_rot, lw=1, color='goldenrod', label='L')
ax2.plot(phase, totV_rot, lw=1, color='indianred', label='V')

ax2.set_ylabel("Flux")
ax2.grid(True, alpha=0.3)
ax2.legend()

# ======================================
# 3) Spectral index
# ======================================
phase2 = np.linspace(0, 1, 1024, endpoint=False)

mask_bins = (alpha_errs < 0.5) & np.isfinite(alphas)
phase_shifted2 = (phase2[mask_bins] - phase_rot) % 1.0

ax3.errorbar(
    phase_shifted2, alphas[mask_bins],
    yerr=alpha_errs[mask_bins],
    fmt='o', ecolor='k',
    alpha=0.5, zorder=2, ms=3
)

ax3.set_xlabel("Phase")
ax3.set_ylabel("Spectral index")
ax3.grid(True, alpha=0.3)

plt.show()